# Data Check: Lead-Lag Features

Quick visual check for EWLL lead-lag pipeline data quality.

1. Run the LeadLag pipeline stage to populate parquet data
2. Load and inspect the output features
3. Plot key features for visual inspection

## Setup

In [ ]:
import sys
sys.path.insert(0, '../../src')

import logging
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from qate.core.model import ExchangeName, Market
from qate.core.symbol import Symbol
from qate.env import sys_env
from qate.store.parquet_store import ParquetStore
from qate.util.dt_range import DtRange

from pipeline.market_pair import EXECUTION_SIZE, LEAD_LAG_PAIRS
from lead_lag.lead_lag import LEAD_LAG_CONFIG_NEARHFT

pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)
sns.set_style('whitegrid')
%matplotlib inline

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(name)s %(levelname)s %(message)s')

POPULATE_DATA = False

PAIR_KEY = 'btc'
MARKET_LEAD, MARKET_LAG = LEAD_LAG_PAIRS[PAIR_KEY]
DATA_START_DATE = '20260101'
DATA_END_DATE = '20260101'

DT_RANGE = DtRange.from_strings(DATA_START_DATE, DATA_END_DATE)
EXECUTION_SIZE_STR = EXECUTION_SIZE.get(MARKET_LAG.symbol)
CONFIG = LEAD_LAG_CONFIG_NEARHFT

print(f'Leader:         {MARKET_LEAD.id}')
print(f'Target (lag):   {MARKET_LAG.id}')
print(f'Date range:     {DATA_START_DATE} - {DATA_END_DATE}')
print(f'Execution size: {EXECUTION_SIZE_STR}')

## 1. Run Pipeline

In [ ]:
from qate.store.market_data_store import MarketDataStore

from pipeline.framework import Pipeline
from pipeline.stages import LeadLagStage
from pipeline.lead_lag_steps import LeadLagStep

def populate_data():
    market_data_store = MarketDataStore(sys_env.get_market_data_dir())
    parquet_store = ParquetStore(sys_env.get_parquet_root_dir())

    pipeline = Pipeline(market_data_store, parquet_store)
    pipeline.stage(
        LeadLagStage(MARKET_LEAD, MARKET_LAG, EXECUTION_SIZE_STR)
        .step(LeadLagStep(CONFIG))
    )

    print('Running pipeline...')
    pipeline.process(DT_RANGE)
    print('Done.')

if POPULATE_DATA:
    populate_data()

## 2. Load & Inspect Parquet Data

In [ ]:
store = ParquetStore(sys_env.get_parquet_root_dir())
ll_store = store.create_lead_lag(MARKET_LEAD, MARKET_LAG, CONFIG.param_id())
df = ll_store.read(DT_RANGE)

print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
print(f'NaN counts:\n{df.isna().sum()}')
display(df.describe())
display(df.head(10))

## 3. Plot Key Features

In [ ]:
# Convert ts to datetime for plotting
df['datetime'] = pd.to_datetime(df['ts'], unit='s')

# Find all max_corr columns
corr_cols = [c for c in df.columns if c.startswith('ll_max_corr')]
lag_cols = [c for c in df.columns if c.startswith('ll_opt_lag_ms')]
gain_cols = [c for c in df.columns if c.startswith('ll_alpha_gain')]
stability_cols = [c for c in df.columns if c.startswith('ll_lag_stability')]
drift_cols = [c for c in df.columns if c.startswith('ll_drift')]

print(f'Max corr columns: {corr_cols}')
print(f'Opt lag columns:  {lag_cols}')
print(f'Alpha gain cols:  {gain_cols}')
print(f'Stability cols:   {stability_cols}')
print(f'Drift cols:       {drift_cols}')

In [ ]:
# Plot ll_max_corr
if corr_cols:
    fig, axes = plt.subplots(len(corr_cols), 1, figsize=(15, 3 * len(corr_cols)), sharex=True)
    if len(corr_cols) == 1:
        axes = [axes]
    for ax, col in zip(axes, corr_cols):
        ax.plot(df['datetime'], df[col], linewidth=0.5)
        ax.set_ylabel(col, fontsize=8)
        ax.axhline(y=0, color='black', linewidth=0.5, linestyle='--')
    axes[0].set_title(f'{MARKET_LEAD.id} -> {MARKET_LAG.id} Max Correlation')
    plt.tight_layout()
    plt.show()

In [ ]:
# Plot ll_opt_lag_ms (normalized)
if lag_cols:
    fig, axes = plt.subplots(len(lag_cols), 1, figsize=(15, 3 * len(lag_cols)), sharex=True)
    if len(lag_cols) == 1:
        axes = [axes]
    for ax, col in zip(axes, lag_cols):
        ax.plot(df['datetime'], df[col], linewidth=0.5, color='orange')
        ax.set_ylabel(col, fontsize=8)
    axes[0].set_title(f'{MARKET_LEAD.id} -> {MARKET_LAG.id} Optimal Lag (normalized)')
    plt.tight_layout()
    plt.show()

In [ ]:
# Plot ll_alpha_gain
if gain_cols:
    fig, axes = plt.subplots(len(gain_cols), 1, figsize=(15, 3 * len(gain_cols)), sharex=True)
    if len(gain_cols) == 1:
        axes = [axes]
    for ax, col in zip(axes, gain_cols):
        ax.plot(df['datetime'], df[col], linewidth=0.5, color='green')
        ax.set_ylabel(col, fontsize=8)
        ax.axhline(y=1, color='red', linewidth=0.5, linestyle='--')
    axes[0].set_title(f'{MARKET_LEAD.id} -> {MARKET_LAG.id} Alpha Gain')
    plt.tight_layout()
    plt.show()

In [ ]:
# Plot ll_lag_stability and ll_drift
if stability_cols:
    fig, axes = plt.subplots(len(stability_cols), 1, figsize=(15, 3 * len(stability_cols)), sharex=True)
    if len(stability_cols) == 1:
        axes = [axes]
    for ax, col in zip(axes, stability_cols):
        ax.plot(df['datetime'], df[col], linewidth=0.5, color='purple')
        ax.set_ylabel(col, fontsize=8)
    axes[0].set_title(f'{MARKET_LEAD.id} -> {MARKET_LAG.id} Lag Stability (log1p)')
    plt.tight_layout()
    plt.show()

if drift_cols:
    fig, axes = plt.subplots(len(drift_cols), 1, figsize=(15, 3 * len(drift_cols)), sharex=True)
    if len(drift_cols) == 1:
        axes = [axes]
    for ax, col in zip(axes, drift_cols):
        ax.plot(df['datetime'], df[col], linewidth=0.5, color='brown')
        ax.set_ylabel(col, fontsize=8)
        ax.axhline(y=0, color='black', linewidth=0.5, linestyle='--')
    axes[0].set_title(f'{MARKET_LEAD.id} -> {MARKET_LAG.id} Lag Drift')
    plt.tight_layout()
    plt.show()

## 4. Correlation Heatmap

In [ ]:
# Correlation heatmap across all lead-lag features
ll_cols = [c for c in df.columns if c.startswith('ll_')]
if ll_cols:
    fig, ax = plt.subplots(figsize=(max(10, len(ll_cols) * 0.6), max(8, len(ll_cols) * 0.5)))
    corr_matrix = df[ll_cols].corr()
    sns.heatmap(corr_matrix, annot=False, cmap='RdBu_r', center=0, ax=ax,
                xticklabels=True, yticklabels=True)
    ax.set_title('Lead-Lag Feature Correlation Heatmap')
    plt.xticks(fontsize=6, rotation=90)
    plt.yticks(fontsize=6)
    plt.tight_layout()
    plt.show()

## 5. Data Consistency Summary

In [ ]:
print('=' * 60)
print('Data Consistency Summary')
print('=' * 60)

print(f'\n--- LeadLag Features ---')
print(f'  Leader:     {MARKET_LEAD.id}')
print(f'  Target:     {MARKET_LAG.id}')
print(f'  Rows:       {len(df):,}')
print(f'  Columns:    {len(df.columns)}')
ts_min = pd.to_datetime(df['ts'].min(), unit='s') if not df.empty else 'N/A'
ts_max = pd.to_datetime(df['ts'].max(), unit='s') if not df.empty else 'N/A'
print(f'  Time range: {ts_min} ~ {ts_max}')
total_nan = df.isna().sum().sum()
print(f'  Total NaN:  {total_nan:,} ({total_nan / max(df.size, 1) * 100:.2f}%)')
dupes = df.duplicated(subset=['ts']).sum()
print(f'  Duplicate timestamps: {dupes}')

# Sanity checks on correlation values
print(f'\n--- Sanity Checks ---')
for col in corr_cols:
    vals = df[col].dropna()
    in_range = ((vals >= -1) & (vals <= 1)).all()
    print(f'  {col} in [-1, 1]: {in_range}')

for col in lag_cols:
    vals = df[col].dropna()
    in_range = ((vals >= 0) & (vals <= 1)).all()
    print(f'  {col} in [0, 1]: {in_range}')